### 📊 **Exercise 10 — Interactive Dashboard**

#### 1️⃣ Install & Import Dependencies (run once)

In [27]:
!pip install pandas plotly ipywidgets -q

In [28]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from ipywidgets import interact, widgets
import numpy as np
import warnings
warnings.filterwarnings('ignore')

#### 2️⃣ Create Synthetic Dataset (100 Transactions)

In [29]:
np.random.seed(42)

n = 100

df = pd.DataFrame({
    "customer_id": np.random.randint(1, 51, n),
    "transaction_id": [f"T{i:04d}" for i in range(1, n + 1)],
    "transaction_date": pd.date_range("2023-01-01", periods=n, freq="D"),
    "transaction_amount": np.random.randint(10, 300, n),
    "product_category": np.random.choice(
        ["Electronics", "Clothing", "Grocery"], n
    ),
    "payment_method": np.random.choice(
        ["Credit Card", "Debit Card", "Cash"], n
    )
})

df["month"] = df["transaction_date"].dt.to_period("M").astype(str)

print(f"Dataset Shape: {df.shape}")
print("\nFirst 5 Rows:")
display(df.head())

Dataset Shape: (100, 7)

First 5 Rows:


,customer_id,transaction_id,transaction_date,transaction_amount,product_category,payment_method,month
0,39,T0001,2023-01-01,18,Electronics,Cash,2023-01
1,29,T0002,2023-01-02,138,Clothing,Debit Card,2023-01
2,15,T0003,2023-01-03,145,Clothing,Credit Card,2023-01
3,43,T0004,2023-01-04,72,Grocery,Debit Card,2023-01
4,8,T0005,2023-01-05,148,Grocery,Credit Card,2023-01


#### 3️⃣ KPI Metrics (Top Summary Cards)

In [30]:
total_transactions = len(df)
total_revenue = df["transaction_amount"].sum()
avg_transaction = df["transaction_amount"].mean()
repeat_customers = df["customer_id"].duplicated().sum()
unique_customers = df["customer_id"].nunique()
avg_revenue_per_customer = total_revenue / unique_customers

# Create a styled metrics display
print("📊 KEY PERFORMANCE INDICATORS")
print("=" * 40)
print(f"Total Transactions: {total_transactions:,}")
print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Average Transaction: ${avg_transaction:,.2f}")
print(f"Repeat Customers: {repeat_customers:,}")
print(f"Unique Customers: {unique_customers:,}")
print(f"Avg Revenue per Customer: ${avg_revenue_per_customer:,.2f}")
print("=" * 40)

# Display as DataFrame too
metrics_df = pd.DataFrame({
    "Metric": [
        "Total Transactions",
        "Total Revenue",
        "Average Transaction",
        "Repeat Customers",
        "Unique Customers",
        "Avg Revenue per Customer"
    ],
    "Value": [
        f"{total_transactions:,}",
        f"${total_revenue:,.2f}",
        f"${avg_transaction:,.2f}",
        f"{repeat_customers:,}",
        f"{unique_customers:,}",
        f"${avg_revenue_per_customer:,.2f}"
    ]
})

print("\n📋 Metrics Table:")
display(metrics_df)

📊 KEY PERFORMANCE INDICATORS
Total Transactions: 100
Total Revenue: $15,815.00
Average Transaction: $158.15
Repeat Customers: 54
Unique Customers: 46
Avg Revenue per Customer: $343.80

📋 Metrics Table:


,Metric,Value
0,Total Transactions,100
1,Total Revenue,"$15,815.00"
2,Average Transaction,$158.15
3,Repeat Customers,54
4,Unique Customers,46
5,Avg Revenue per Customer,$343.80


#### 4️⃣ Initial Data Exploration Visualizations

In [31]:
print("🔍 Initial Data Overview Visualizations")
print("-" * 40)

# 1. Overall transaction amount distribution
fig1 = px.histogram(df, x="transaction_amount", 
                   nbins=20,
                   title="Overall Transaction Amount Distribution",
                   color_discrete_sequence=['#2E86AB'])
fig1.update_layout(showlegend=False)
fig1.show()

# 2. Payment method distribution
fig2 = px.pie(df, names="payment_method", 
              title="Overall Payment Method Distribution",
              hole=0.3,
              color_discrete_sequence=px.colors.qualitative.Set3)
fig2.show()

# 3. Monthly sales trend
monthly_sales = df.groupby("month")["transaction_amount"].sum().reset_index()
fig3 = px.line(monthly_sales, x="month", y="transaction_amount",
              markers=True,
              title="Overall Monthly Sales Trend",
              line_shape="spline")
fig3.update_traces(line=dict(width=3))
fig3.show()

# 4. Product category distribution
category_counts = df["product_category"].value_counts().reset_index()
category_counts.columns = ["category", "count"]  # rename columns for clarity

fig4 = px.bar(
    category_counts,
    x="category",
    y="count",
    title="Overall Transactions by Product Category",
    labels={"category": "Product Category", "count": "Number of Transactions"},
    color="category",
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig4.show()


🔍 Initial Data Overview Visualizations
----------------------------------------


#### 5️⃣ Enhanced Interactive Dashboard Function

In [32]:
def enhanced_dashboard(category="All", payment="All", show_metrics=True):
    """
    Enhanced dashboard with multiple visualizations and metrics
    """
    # Filter data
    filtered = df.copy()
    
    if category != "All":
        filtered = filtered[filtered["product_category"] == category]
    
    if payment != "All":
        filtered = filtered[filtered["payment_method"] == payment]
    
    # Calculate filtered metrics
    filtered_transactions = len(filtered)
    filtered_revenue = filtered["transaction_amount"].sum()
    filtered_avg = filtered["transaction_amount"].mean() if len(filtered) > 0 else 0
    
    # Display metrics
    if show_metrics:
        print("📊 FILTERED METRICS")
        print("=" * 40)
        print(f"Active Filters: Category={category}, Payment={payment}")
        print(f"Filtered Transactions: {filtered_transactions:,}")
        print(f"Filtered Revenue: ${filtered_revenue:,.2f}")
        print(f"Filtered Average: ${filtered_avg:,.2f}")
        print("=" * 40 + "\n")
    
    if len(filtered) == 0:
        print("⚠️ No data available for selected filters!")
        return
    
    # Create subplots - 2x2 grid
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "Transaction Amount Distribution",
            "Payment Method Distribution",
            "Monthly Sales Trend",
            "Product Category Distribution"
        ),
        specs=[[{"type": "histogram"}, {"type": "pie"}],
               [{"type": "scatter"}, {"type": "bar"}]]
    )
    
    # 1. Histogram
    hist = go.Histogram(
        x=filtered["transaction_amount"],
        nbinsx=20,
        name="Amount",
        marker_color='#2E86AB'
    )
    fig.add_trace(hist, row=1, col=1)
    
    # 2. Pie chart
    if payment == "All":
        payment_counts = filtered["payment_method"].value_counts()
        pie = go.Pie(
            labels=payment_counts.index,
            values=payment_counts.values,
            name="Payment",
            hole=0.3,
            marker_colors=['#A23B72', '#F18F01', '#C73E1D']
        )
        fig.add_trace(pie, row=1, col=2)
    else:
        # If payment method is filtered, show product category instead
        cat_counts = filtered["product_category"].value_counts()
        pie = go.Pie(
            labels=cat_counts.index,
            values=cat_counts.values,
            name="Category",
            hole=0.3,
            marker_colors=['#2E86AB', '#A23B72', '#F18F01']
        )
        fig.add_trace(pie, row=1, col=2)
    
    # 3. Line chart
    monthly = filtered.groupby("month")["transaction_amount"].sum().reset_index()
    line = go.Scatter(
        x=monthly["month"],
        y=monthly["transaction_amount"],
        mode="lines+markers",
        name="Sales",
        line=dict(color='#C73E1D', width=3)
    )
    fig.add_trace(line, row=2, col=1)
    
    # 4. Bar chart
    if category == "All":
        bar_data = filtered["product_category"].value_counts().reset_index()
        bar_data.columns = ["category", "count"]  # 🔑 Rename columns to avoid KeyError
        bar = go.Bar(
            x=bar_data["category"],
            y=bar_data["count"],
            name="Categories",
            marker_color=['#2E86AB', '#A23B72', '#F18F01']
        )
        fig.add_trace(bar, row=2, col=2)
    else:
        payment_data = filtered["payment_method"].value_counts().reset_index()
        payment_data.columns = ["payment", "count"]  # 🔑 Rename columns
        bar = go.Bar(
            x=payment_data["payment"],
            y=payment_data["count"],
            name="Payments",
            marker_color=['#A23B72', '#F18F01', '#C73E1D']
        )
        fig.add_trace(bar, row=2, col=2)
    
    # Update layout
    fig.update_layout(
        height=800,
        showlegend=False,
        title_text=f"Interactive Dashboard | Filters: Category={category}, Payment={payment}",
        title_font_size=20
    )
    
    fig.update_xaxes(title_text="Amount ($)", row=1, col=1)
    fig.update_yaxes(title_text="Count", row=1, col=1)
    fig.update_xaxes(title_text="Month", row=2, col=1)
    fig.update_yaxes(title_text="Sales ($)", row=2, col=1)
    fig.update_xaxes(title_text="Category", row=2, col=2)
    fig.update_yaxes(title_text="Count", row=2, col=2)
    
    fig.show()
    
    # Display data summary
    print("📋 Filtered Data Summary:")
    print(f"Date Range: {filtered['transaction_date'].min().date()} to {filtered['transaction_date'].max().date()}")
    print(f"Total Amount: ${filtered['transaction_amount'].sum():,.2f}")
    print(f"Transaction Count: {len(filtered):,}")
    print("-" * 40)


#### 6️⃣ Activate Interactive Controls

In [ ]:
# ----------------------------
# CREATE INTERACTIVE WIDGETS
# ----------------------------
print("🎛️ INTERACTIVE DASHBOARD CONTROLS")
print("=" * 50)
print("Adjust the filters below to update all visualizations in real-time!")
print("=" * 50 + "\n")

# Dropdowns and toggle
category_dropdown = widgets.Dropdown(
    options=["All", "Electronics", "Clothing", "Grocery"],
    value="All",
    description="📦 Category:",
    style={'description_width': 'initial'}
)

payment_dropdown = widgets.Dropdown(
    options=["All", "Credit Card", "Debit Card", "Cash"],
    value="All",
    description="💳 Payment:",
    style={'description_width': 'initial'}
)

show_metrics_toggle = widgets.Checkbox(
    value=True,
    description="Show Metrics",
    style={'description_width': 'initial'}
)

# Reset button
reset_button = widgets.Button(
    description="🔄 Reset Filters",
    button_style='info',
    layout=widgets.Layout(width='150px')
)

def reset_filters(b):
    category_dropdown.value = "All"
    payment_dropdown.value = "All"
    show_metrics_toggle.value = True

reset_button.on_click(reset_filters)

# Display widgets
display(widgets.VBox([
    widgets.HBox([category_dropdown, payment_dropdown]),
    widgets.HBox([show_metrics_toggle, reset_button])
]))

# Link widgets to dashboard
interact_output = widgets.interactive_output(
    enhanced_dashboard,
    {
        "category": category_dropdown,
        "payment": payment_dropdown,
        "show_metrics": show_metrics_toggle
    }
)

display(interact_output)

🎛️ INTERACTIVE DASHBOARD CONTROLS
Adjust the filters below to update all visualizations in real-time!



Output()

#### 7️⃣ Advanced Analytics Section

In [34]:
print("🔬 ADVANCED ANALYTICS")
print("=" * 40)

# Customer segmentation analysis
customer_analysis = df.groupby("customer_id").agg({
    "transaction_id": "count",
    "transaction_amount": ["sum", "mean", "std"],
    "product_category": lambda x: x.mode()[0] if len(x.mode()) > 0 else "N/A",
    "payment_method": lambda x: x.mode()[0] if len(x.mode()) > 0 else "N/A"
}).round(2)

customer_analysis.columns = [
    "Transaction_Count", "Total_Spent", "Avg_Transaction", 
    "Std_Transaction", "Favorite_Category", "Preferred_Payment"
]

customer_analysis = customer_analysis.sort_values("Total_Spent", ascending=False)

print("🏆 Top 10 Customers by Total Spend:")
display(customer_analysis.head(10))

# Category analysis
category_analysis = df.groupby("product_category").agg({
    "transaction_id": "count",
    "transaction_amount": ["sum", "mean", "min", "max"]
}).round(2)

category_analysis.columns = [
    "Transaction_Count", "Total_Revenue", "Avg_Amount", 
    "Min_Amount", "Max_Amount"
]

print("\n📊 Product Category Performance:")
display(category_analysis)

# Visualization: Customer segments
fig_segments = px.scatter(
    customer_analysis.reset_index(),
    x="Transaction_Count",
    y="Total_Spent",
    size="Avg_Transaction",
    color="Favorite_Category",
    hover_data=["customer_id", "Preferred_Payment"],
    title="Customer Segments Analysis",
    labels={
        "Transaction_Count": "Number of Transactions",
        "Total_Spent": "Total Amount Spent ($)",
        "Avg_Transaction": "Average Transaction Size"
    }
)

fig_segments.update_layout(height=500)
fig_segments.show()

🔬 ADVANCED ANALYTICS
🏆 Top 10 Customers by Total Spend:


,Transaction_Count,Total_Spent,Avg_Transaction,Std_Transaction,Favorite_Category,Preferred_Payment
customer_id,,,,,,
26,3,660,220.00,41.57,Clothing,Debit Card
24,4,629,157.25,112.42,Clothing,Credit Card
44,5,623,124.60,96.95,Clothing,Credit Card
21,4,618,154.50,92.01,Grocery,Cash
14,4,567,141.75,116.38,Clothing,Debit Card
2,4,566,141.50,99.99,Electronics,Credit Card
15,4,565,141.25,55.05,Clothing,Debit Card
4,3,550,183.33,55.08,Clothing,Credit Card
25,4,522,130.50,77.38,Clothing,Cash



📊 Product Category Performance:


,Transaction_Count,Total_Revenue,Avg_Amount,Min_Amount,Max_Amount
product_category,,,,,
Clothing,32,4983,155.72,11,268
Electronics,31,4934,159.16,18,298
Grocery,37,5898,159.41,14,293


#### 8️⃣ Export & Save Functionality

In [35]:
print("💾 EXPORT FUNCTIONALITY")
print("=" * 40)

def export_data(format_type="CSV"):
    """Export the dataset in different formats"""
    timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    
    if format_type == "CSV":
        filename = f"transaction_data_{timestamp}.csv"
        df.to_csv(filename, index=False)
        print(f"✅ Data exported to {filename}")
    elif format_type == "Excel":
        filename = f"transaction_data_{timestamp}.xlsx"
        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            df.to_excel(writer, sheet_name='Transactions', index=False)
            customer_analysis.to_excel(writer, sheet_name='Customer_Analysis')
            category_analysis.to_excel(writer, sheet_name='Category_Analysis')
        print(f"✅ Data exported to {filename} with multiple sheets")
    elif format_type == "JSON":
        filename = f"transaction_data_{timestamp}.json"
        df.to_json(filename, orient='records', indent=2)
        print(f"✅ Data exported to {filename}")
    
    return filename

# Create export buttons
export_csv = widgets.Button(
    description="Export to CSV",
    button_style='success',
    layout=widgets.Layout(width='150px')
)

export_excel = widgets.Button(
    description="Export to Excel",
    button_style='warning',
    layout=widgets.Layout(width='150px')
)

export_json = widgets.Button(
    description="Export to JSON",
    button_style='info',
    layout=widgets.Layout(width='150px')
)

def on_export_csv(b):
    export_data("CSV")

def on_export_excel(b):
    export_data("Excel")

def on_export_json(b):
    export_data("JSON")

export_csv.on_click(on_export_csv)
export_excel.on_click(on_export_excel)
export_json.on_click(on_export_json)

# Display export buttons
print("Click a button to export the data:")
display(widgets.HBox([export_csv, export_excel, export_json]))

💾 EXPORT FUNCTIONALITY
Click a button to export the data:


## ✅ **Dashboard Summary:**

### 🎯 **What This Dashboard Delivers:**

1. **📊 Complete Analytics Pipeline**
   - Data generation & preprocessing
   - KPI metrics calculation
   - Interactive visualizations

2. **🎛️ Interactive Controls**
   - Product category filter
   - Payment method filter
   - Metrics toggle
   - Reset functionality

3. **📈 Multiple Visualization Types**
   - Histograms for distribution analysis
   - Pie charts for composition
   - Line charts for trends
   - Bar charts for comparisons
   - Scatter plots for segmentation

4. **🔬 Advanced Analytics**
   - Customer segmentation
   - Category performance
   - Top customer analysis

5. **💾 Export Capabilities**
   - CSV export
   - Excel export (multi-sheet)
   - JSON export

### 🚀 **Portfolio Ready Features:**
- Professional, clean visual design
- Real-time interactivity
- Comprehensive analytics
- Production-ready code structure
- Export functionality
- Well-documented sections

### 🔥 **Next Steps Available:**
This dashboard can be extended to:
- **Streamlit web application**
- **Predictive analytics** (ML models)
- **Real-time data integration**
- **Advanced customer clustering**
- **Automated reporting system**

In [36]:
# Final summary cell
print("✨ DASHBOARD COMPLETE")
print("=" * 40)
print(f"Total Rows: {len(df):,}")
print(f"Date Range: {df['transaction_date'].min().date()} to {df['transaction_date'].max().date()}")
print(f"Total Revenue: ${df['transaction_amount'].sum():,.2f}")
print(f"Unique Customers: {df['customer_id'].nunique():,}")
print("=" * 40)
print("\n✅ Ready for portfolio presentation! 🚀")

✨ DASHBOARD COMPLETE
Total Rows: 100
Date Range: 2023-01-01 to 2023-04-10
Total Revenue: $15,815.00
Unique Customers: 46

✅ Ready for portfolio presentation! 🚀
